# Futoshiki Vision + CP — Reproducción en Google Colab

Este notebook usa únicamente el repositorio final de GitHub. No requiere subir kits ZIP ni copiar código manualmente.

In [ ]:
GITHUB_USER = "TU_USUARIO"
REPO_NAME = "futoshiki-vision-cp"
BRANCH = "main"

assert GITHUB_USER != "TU_USUARIO", "Coloca tu usuario de GitHub."
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
print(REPO_URL)

In [ ]:
from pathlib import Path
import os

repo_dir = Path('/content') / REPO_NAME
if repo_dir.exists():
    !rm -rf "$repo_dir"

!git clone --depth 1 --branch "$BRANCH" "$REPO_URL" "$repo_dir"
%cd $repo_dir

In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
!sha256sum -c models/checksums.sha256
!pytest -q

In [ ]:
from google.colab import files
from pathlib import Path

print('Sube una imagen 4x4 o 5x5')
uploaded = files.upload()
name, data = next(iter(uploaded.items()))
image_path = Path('/content') / name
image_path.write_bytes(data)
print(image_path)

In [ ]:
SOURCE_TYPE = "printed_photo"  # digital, printed_photo, screen_photo o generated_photo
EXPECTED_N = None  # None autodetecta 4x4 / 5x5


In [ ]:
from futoshiki_assistant.config import VisionSettings
from futoshiki_assistant.pipeline import FutoshikiPipeline
from futoshiki_assistant.vision.models import load_model_bundle

settings = VisionSettings.from_json('config/default.json')
models = load_model_bundle('models')
pipeline = FutoshikiPipeline(models, settings)

result = pipeline.process(
    image_path,
    source_type=SOURCE_TYPE,
    expected_n=EXPECTED_N,
)

print('N detectado:', result['size'])
print('Gate detección:', result['detection_gate'])
print('Segmentación:', result['segmentation_method'])
print('Givens:', len(result['instance']['givens']))
print('Desigualdades:', len(result['instance']['inequalities']))
print('Estado CP:', result['cp_state'])
print('Tiempo total:', round(result['total_time'], 4), 's')
print(result['instance'])

In [ ]:
import matplotlib.pyplot as plt
import cv2

def show_bgr(image, title):
    if image is None:
        return
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

show_bgr(result['board_overlay'], 'Detección de tablero')
show_bgr(result['rectified'], 'Tablero rectificado')
show_bgr(result['segmentation_overlay'], 'Segmentación')
show_bgr(result['perception_overlay'], 'Percepción')
show_bgr(result['solution_overlay'], 'Solución rectificada')
show_bgr(result['final_overlay'], 'Solución sobre imagen original')